# Week 51, Day 5: Grids as graphs

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w051d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import heapq

terrain = ["119",
           "191",
           "111"]

def cheapest(grid):
    rows, cols = len(grid), len(grid[0])
    dist = {(0, 0): 0}
    heap = [(0, 0, 0)]
    while heap:
        d, r, c = heapq.heappop(heap)
        if (r, c) == (rows - 1, cols - 1):
            return d
        if d > dist[(r, c)]:
            continue
        for nr, nc in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)):
            if 0 <= nr < rows and 0 <= nc < cols:
                nd = d + int(grid[nr][nc])
                if nd < dist.get((nr, nc), float("inf")):
                    dist[(nr, nc)] = nd
                    heapq.heappush(heap, (nd, nr, nc))

print(cheapest(terrain))

In [ ]:
import heapq, math

def route_cost(grid, diagonal):
    rows, cols = len(grid), len(grid[0])
    steps = [(-1, 0, 1), (1, 0, 1), (0, -1, 1), (0, 1, 1)]
    if diagonal:
        steps += [(dr, dc, math.sqrt(2)) for dr in (-1, 1) for dc in (-1, 1)]
    dist, heap = {(0, 0): 0}, [(0, 0, 0)]
    while heap:
        d, r, c = heapq.heappop(heap)
        if (r, c) == (rows - 1, cols - 1):
            return round(d, 3)
        if d > dist[(r, c)]:
            continue
        for dr, dc, step in steps:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != "#":
                if dr and dc and (grid[r][nc] == "#" or grid[nr][c] == "#"):
                    continue                          # no cutting corners
                nd = d + step
                if nd < dist.get((nr, nc), float("inf")):
                    dist[(nr, nc)] = nd
                    heapq.heappush(heap, (nd, nr, nc))

field = ["....",
         ".#..",
         "....",
         "...."]
print(route_cost(field, diagonal=False), route_cost(field, diagonal=True))

In [ ]:
import heapq

def search(grid, start, goal, use_heuristic):
    rows, cols = len(grid), len(grid[0])
    def h(cell):
        return abs(cell[0] - goal[0]) + abs(cell[1] - goal[1]) if use_heuristic else 0
    g = {start: 0}
    heap = [(h(start), 0, start)]
    expanded = 0
    while heap:
        f, neg_d, cell = heapq.heappop(heap)
        d = -neg_d
        if d > g[cell]:
            continue
        expanded += 1
        if cell == goal:
            return d, expanded
        r, c = cell
        for nxt in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)):
            if 0 <= nxt[0] < rows and 0 <= nxt[1] < cols and grid[nxt[0]][nxt[1]] != "#":
                if d + 1 < g.get(nxt, float("inf")):
                    g[nxt] = d + 1
                    heapq.heappush(heap, (d + 1 + h(nxt), -(d + 1), nxt))   # ties: further along first
    return None, expanded

grid = ["." * 40 for _ in range(40)]
grid[20] = "#" * 30 + "." * 10
print("Dijkstra:", search(grid, (0, 0), (39, 0), False))
print("A*:      ", search(grid, (0, 0), (39, 0), True))

In [ ]:
def bidirectional(neighbours, start, goal):
    if start == goal:
        return 0
    front, back = {start}, {goal}
    seen_f, seen_b = {start: 0}, {goal: 0}
    while front and back:
        if len(front) > len(back):                   # grow the smaller side
            front, back, seen_f, seen_b = back, front, seen_b, seen_f
        nxt = set()
        for v in front:
            for w in neighbours(v):
                if w in seen_b:
                    return seen_f[v] + 1 + seen_b[w]
                if w not in seen_f:
                    seen_f[w] = seen_f[v] + 1
                    nxt.add(w)
        front = nxt
    return None

def moves(n):          # an implicit graph: from n you can go to n + 1, n - 1 or 2n
    return [n + 1, n - 1, 2 * n]

print(bidirectional(moves, 3, 100), bidirectional(moves, 5, 5))

In [ ]:
from collections import deque

def with_keys(grid):
    rows, cols = len(grid), len(grid[0])
    start = next((r, c) for r in range(rows) for c in range(cols) if grid[r][c] == "S")
    state = (start, 0)
    dist = {state: 0}
    queue = deque([state])
    while queue:
        (r, c), keys = queue.popleft()
        if grid[r][c] == "E":
            return dist[((r, c), keys)]
        for nr, nc in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)):
            if not (0 <= nr < rows and 0 <= nc < cols):
                continue
            ch, new_keys = grid[nr][nc], keys
            if ch == "#":
                continue
            if ch.isupper() and ch not in "SE" and not keys & (1 << (ord(ch) - ord("A"))):
                continue                              # a locked door
            if ch.islower():
                new_keys = keys | (1 << (ord(ch) - ord("a")))
            nxt = ((nr, nc), new_keys)
            if nxt not in dist:
                dist[nxt] = dist[((r, c), keys)] + 1
                queue.append(nxt)
    return None

castle = ["S.A.E",
          ".####",
          "....a"]
print(with_keys(castle))

> **Key idea.** A grid is an implicit graph; terrain costs make it weighted, so use Dijkstra's algorithm with a neighbours function, and cost diagonal moves √2 (forbidding cut corners if needed). A\* orders the heap by g + h, where h estimates the remaining cost; if h never overestimates (Manhattan distance for 4-way moves, octile for 8-way) it still finds a shortest path while exploring far less, h = 0 being Dijkstra's algorithm. Bidirectional BFS grows the smaller frontier from each end until they meet, exploring about 2 × b^(d/2) instead of b^d. When the next move depends on more than the cell, search states like (cell, keys as a bitmask) or (cell, facing).

### Exercise: Terrain

A grid's digits are the cost of stepping **onto** each cell (the start's own cost isn't paid); `#` is impassable. Write `cheapest(grid, start, goal)`: the least cost from start to goal moving in 4 directions, or `None`. Then `cost_map(grid, start)`: a list of lists of the least cost to every cell (`None` for walls and unreachable cells).

In [ ]:
import heapq

def cheapest(grid, start, goal):
    ___

def cost_map(grid, start):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_terrain():\n    \"terrain\"\n    t = [\"119\", \"191\", \"111\"]\n    assert cheapest(t, (0, 0), (2, 2)) == 4 and cheapest(t, (0, 0), (0, 0)) == 0\n    assert cheapest([\"1#1\"], (0, 0), (0, 2)) is None\n    assert cost_map([\"12\", \"#3\"], (0, 0)) == [[0, 2], [None, 5]]\n    rng = random.Random(1)\n    for _ in range(30):\n        g = [\"\".join(rng.choice(\"1239#\") for _ in range(5)) for _ in range(5)]\n        g[0] = \"1\" + g[0][1:]\n        m = cost_map(g, (0, 0))\n        for r in range(5):\n            for c in range(5):\n                if m[r][c] is not None and (r, c) != (0, 0):\n                    best = min(m[nr][nc] for nr, nc in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1))\n                               if 0 <= nr < 5 and 0 <= nc < 5 and m[nr][nc] is not None)\n                    assert m[r][c] == best + int(g[r][c])")

### Exercise: A* search

Write `astar(grid, start, goal)` for a grid of `.` and `#` with 4-way moves of cost 1: a pair `(length, expanded)`, the shortest path length (or `None`) and the number of cells **expanded** (popped and not out of date), using the Manhattan heuristic and breaking ties in f towards the **larger** cost so far. Then `dijkstra_expanded(grid, start, goal)`: the same count with h = 0. On open grids A* must expand far fewer.

In [ ]:
import heapq

def astar(grid, start, goal):
    ___

def dijkstra_expanded(grid, start, goal):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nfrom collections import deque\ndef bfs(grid, s, t):\n    rows, cols = len(grid), len(grid[0])\n    d = {s: 0}\n    q = deque([s])\n    while q:\n        r, c = q.popleft()\n        for nr, nc in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)):\n            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] != \"#\" and (nr, nc) not in d:\n                d[(nr, nc)] = d[(r, c)] + 1\n                q.append((nr, nc))\n    return d.get(t)\ndef test_astar():\n    \"A*\"\n    open_grid = [\".\" * 30 for _ in range(30)]\n    length, exp = astar(open_grid, (0, 0), (29, 29))\n    assert length == 58 and exp < dijkstra_expanded(open_grid, (0, 0), (29, 29)) // 5\n    assert astar([\".#.\"], (0, 0), (0, 2))[0] is None\n    rng = random.Random(2)\n    for _ in range(100):\n        g = [\"\".join(rng.choice(\"...#\") for _ in range(8)) for _ in range(8)]\n        g[0] = \".\" + g[0][1:]\n        g[7] = g[7][:7] + \".\"\n        assert astar(g, (0, 0), (7, 7))[0] == bfs(g, (0, 0), (7, 7))")

### Exercise: From both ends

Write `bidirectional(neighbours, start, goal)`: the shortest distance in an undirected implicit graph (where `neighbours(v)` returns a list, and moves are reversible), or `None`, growing the smaller frontier one level at a time. Then `count_expanded(neighbours, start, goal)`: how many vertices it **adds** to either side before meeting, so it can be compared with plain BFS.

In [ ]:
def bidirectional(neighbours, start, goal):
    ___

def count_expanded(neighbours, start, goal):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from collections import deque\ndef bfs(neighbours, s, t, limit=10**6):\n    d = {s: 0}\n    q = deque([s])\n    while q:\n        v = q.popleft()\n        if v == t:\n            return d[v], len(d)\n        for w in neighbours(v):\n            if w not in d:\n                d[w] = d[v] + 1\n                q.append(w)\n    return None, len(d)\ndef grid_n(cell):\n    r, c = cell\n    return [(r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)]\ndef test_bidir():\n    \"both ends\"\n    assert bidirectional(grid_n, (0, 0), (0, 0)) == 0 and bidirectional(grid_n, (0, 0), (3, 4)) == 7\n    for t in [(5, 5), (10, -3), (0, 12)]:\n        d, size = bfs(grid_n, (0, 0), t)\n        assert bidirectional(grid_n, (0, 0), t) == d and count_expanded(grid_n, (0, 0), t) < size\n    line = lambda n: [x for x in (n - 1, n + 1) if 0 <= x <= 10]\n    assert bidirectional(line, 0, 10) == 10 and bidirectional(lambda n: [], 1, 2) is None")

### Exercise: Keys and doors

A grid has `S` (start), `E` (exit), `#` walls, `.` floor, lower-case keys `a` to `d` and matching upper-case doors `A` to `D`. Write `fewest_steps(grid)`: the fewest moves from S to E, where a door can be passed only while holding its key (picked up by stepping on it; keys aren't used up), or `None`. BFS over `(cell, keys bitmask)`. Then `keys_needed(grid)`: the sorted list of key letters held on arriving at E along one shortest route.

In [ ]:
from collections import deque

def fewest_steps(grid):
    ___

def keys_needed(grid):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_keys():\n    \"keys\"\n    castle = [\"S.A.E\",\n              \".####\",\n              \"....a\"]\n    assert fewest_steps(castle) == 16 and keys_needed(castle) == [\"a\"]\n    assert fewest_steps([\"S.E\"]) == 2 and keys_needed([\"S.E\"]) == []\n    assert fewest_steps([\"SAE\", \"##a\"]) is None\n    two = [\"S.b.A..E\",\n           \".######.\",\n           \"a......B\"]\n    assert fewest_steps(two) is not None and keys_needed(two) == [\"a\", \"b\"]")

### Exercise: Fewest turns

A robot in a grid of `.` and `#` moves one cell forward at a time and can turn left or right on the spot (each turn is a quarter turn). It starts at `start` facing a given direction (`"N"`, `"E"`, `"S"` or `"W"`), and any final direction is fine. Write `fewest_turns(grid, start, facing, goal)`: the fewest quarter-turns needed to reach the goal (moving forward is free), or `None`, with 0-1 BFS over `(cell, direction)` states. Then `cheapest_drive(grid, start, facing, goal, move_cost, turn_cost)`: the least total cost with Dijkstra's algorithm over the same states.

In [ ]:
import heapq
from collections import deque

def fewest_turns(grid, start, facing, goal):
    ___

def cheapest_drive(grid, start, facing, goal, move_cost, turn_cost):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_turns():\n    \"turns\"\n    g = [\"....\",\n         \".##.\",\n         \"....\"]\n    assert fewest_turns(g, (0, 0), \"E\", (2, 3)) == 1\n    assert fewest_turns(g, (0, 0), \"W\", (0, 3)) == 2 and fewest_turns(g, (0, 0), \"N\", (0, 0)) == 0\n    assert fewest_turns([\".#.\"], (0, 0), \"E\", (0, 2)) is None\n    assert cheapest_drive(g, (0, 0), \"E\", (2, 3), 1, 0) == 5 and cheapest_drive(g, (0, 0), \"E\", (2, 3), 1, 10) == 15\n    assert cheapest_drive(g, (0, 0), \"S\", (2, 3), 1, 10) == 15")

### Bug hunt: An overconfident estimate

This A\* multiplies the Manhattan distance by 3, so the heuristic **overestimates** and the search rushes towards the goal, settling for a longer route: this prints `14`. Use the plain Manhattan distance so it finds the true shortest, `10`.

In [ ]:
import heapq

grid = ["...#..#",
        ".#....#",
        ".#.#.#.",
        ".#..#..",
        "......."]

def astar(start, goal):
    rows, cols = len(grid), len(grid[0])
    h = lambda cell: 3 * (abs(cell[0] - goal[0]) + abs(cell[1] - goal[1]))
    g, heap = {start: 0}, [(h(start), 0, start)]
    while heap:
        f, d, cell = heapq.heappop(heap)
        if cell == goal:
            return d
        if d > g[cell]:
            continue
        r, c = cell
        for nxt in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)):
            if 0 <= nxt[0] < rows and 0 <= nxt[1] < cols and grid[nxt[0]][nxt[1]] != "#":
                if d + 1 < g.get(nxt, float("inf")):
                    g[nxt] = d + 1
                    heapq.heappush(heap, (d + 1 + h(nxt), d + 1, nxt))

print(astar((2, 0), (2, 6)))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"never overestimate\"\n    expect_output(\"10\")")

### Bug hunt: Forgetting the keys

This search records only the **cell** as seen, not the keys held, so after walking past the start without the key it can never come back with it: this prints `None`. Make the state `(cell, keys)` so it prints `16`.

In [ ]:
from collections import deque

castle = ["S.A.E",
          ".####",
          "....a"]
start = (0, 0)
seen = {start}
queue = deque([(start, False, 0)])
answer = None
while queue:
    (r, c), has_key, steps = queue.popleft()
    if castle[r][c] == "E":
        answer = steps
        break
    for nr, nc in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)):
        if 0 <= nr < 3 and 0 <= nc < 5 and castle[nr][nc] != "#":
            if castle[nr][nc] == "A" and not has_key:
                continue
            key = has_key or castle[nr][nc] == "a"
            if (nr, nc) not in seen:
                seen.add((nr, nc))
                queue.append(((nr, nc), key, steps + 1))
print(answer)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"keys are part of the state\"\n    expect_output(\"16\")")

### Bug hunt: Diagonals for free

This route planner lets diagonal steps cost 1, the same as straight ones, so it reports a 4 × 4 field crossed corner to corner in 3 moves' worth of distance: this prints `3`. Charge `math.sqrt(2)` for a diagonal step so it prints `4.243`.

In [ ]:
import heapq, math

n = 4
steps = [(dr, dc, 1) for dr in (-1, 0, 1) for dc in (-1, 0, 1) if (dr, dc) != (0, 0)]
dist, heap = {(0, 0): 0}, [(0, (0, 0))]
while heap:
    d, (r, c) = heapq.heappop(heap)
    if d > dist[(r, c)]:
        continue
    for dr, dc, step in steps:
        nr, nc = r + dr, c + dc
        if 0 <= nr < n and 0 <= nc < n and d + step < dist.get((nr, nc), float("inf")):
            dist[(nr, nc)] = d + step
            heapq.heappush(heap, (d + step, (nr, nc)))
print(round(dist[(n - 1, n - 1)], 3))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"root two\"\n    expect_output(\"4.243\")")

### Puzzle: An A* step

Arrange the lines into the heart of A* search. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the Manhattan distance from (1, 2) to (4, 0)
q = ___   # the octile distance from (0, 0) to (3, 1), rounded to 3 places
r = ___   # A* with h = 0 is the same as which algorithm? (a string)
s = ___   # the bitmask for holding keys a and c (a is bit 0)
t = ___   # how many (cell, facing) states a 5 x 4 grid with no walls has

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 5 and q == 3.414 and r.lower().startswith(\"dijkstra\"), \"p, q and r\"\n    assert s == 5 and t == 80, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the sliding-tile puzzle

The 8-puzzle has tiles 1 to 8 and a gap (0) on a 3 × 3 board; a move slides a tile next to the gap into it. Solve it with A\*: each state is a tuple of 9 numbers (row by row), each move costs 1, and the heuristic is the sum of the **Manhattan distances** of each tile (not the gap) from its place in the goal `(1, 2, 3, 4, 5, 6, 7, 8, 0)`. That heuristic never overestimates, since each move shifts one tile one square.

Half of all arrangements can't be solved. A state is solvable exactly when the number of **inversions** (pairs of tiles, ignoring the gap, where a larger number comes before a smaller one in the row-by-row order) is even. Write `solvable(state)`, `solve(state)` (the fewest moves, or `None` if unsolvable, without searching), and `solution(state)`: the list of states from the start to the goal along a shortest solution. The checks include positions needing over 20 moves; plain BFS would have to look at up to 181,440 states, but A* looks at far fewer.

In [ ]:
import heapq

GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)

def solvable(state):
    ___

def solve(state):
    ___

def solution(state):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\nfrom collections import deque\nGOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)\ndef neighbours(s):\n    i = s.index(0)\n    r, c = divmod(i, 3)\n    out = []\n    for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):\n        nr, nc = r + dr, c + dc\n        if 0 <= nr < 3 and 0 <= nc < 3:\n            j = nr * 3 + nc\n            t = list(s)\n            t[i], t[j] = t[j], t[i]\n            out.append(tuple(t))\n    return out\ndef test_small():\n    \"small cases\"\n    assert solve(GOAL) == 0 and solution(GOAL) == [GOAL]\n    s = (1, 2, 3, 4, 5, 6, 7, 0, 8)\n    assert solve(s) == 1 and solution(s) == [s, GOAL]\n    assert not solvable((2, 1, 3, 4, 5, 6, 7, 8, 0)) and solve((2, 1, 3, 4, 5, 6, 7, 8, 0)) is None\ndef test_random():\n    \"against BFS\"\n    rng = random.Random(6)\n    for _ in range(6):\n        s = GOAL\n        for _ in range(rng.randint(5, 30)):\n            s = rng.choice(neighbours(s))\n        d = {s: 0}\n        q = deque([s])\n        while q:\n            v = q.popleft()\n            if v == GOAL:\n                break\n            for w in neighbours(v):\n                if w not in d:\n                    d[w] = d[v] + 1\n                    q.append(w)\n        assert solve(s) == d[GOAL]\n        path = solution(s)\n        assert path[0] == s and path[-1] == GOAL and len(path) == d[GOAL] + 1 and all(b in neighbours(a) for a, b in zip(path, path[1:]))\ndef test_hard():\n    \"a hard one, quickly\"\n    t0 = time.perf_counter()\n    assert solve((8, 6, 7, 2, 5, 4, 3, 0, 1)) == 31\n    assert time.perf_counter() - t0 < 20")

## Warm-up

Each digit is the cost of stepping **onto** that cell. The cheap route goes down the left side and along the bottom (1 + 1 + 1 + 1 = 4), avoiding the 9s, though a route through the middle would be just as short in steps. Here `if d > dist[...]: continue` plays the part of the `done` set: an entry whose distance is bigger than the best known is out of date.

## Terrain

A grid with different costs (marsh, road, hill) is a weighted implicit graph: each cell is a vertex, each move an edge whose weight depends on the cell entered (or both cells, or the direction). Dijkstra's algorithm (Monday) works unchanged with a `neighbours` function.

Allowing **diagonal** moves changes the geometry: a diagonal step covers √2 ≈ 1.414 times the distance of a straight one, so it should cost more. Counting it as 1 makes paths zig-zag oddly; using √2 (or 14 against 10 for straight moves, to stay with whole numbers) gives natural routes. Many games also forbid **cutting corners**: a diagonal move is only allowed if both cells beside it are open.

## A* search

Dijkstra's algorithm spreads out in all directions like a ripple, exploring cells far behind the start just as eagerly as cells towards the goal. **A\* search** (Peter Hart, Nils Nilsson and Bertram Raphael, 1968, for a robot called Shakey) steers it: it orders the heap by

f(v) = g(v) + h(v)

where g(v) is the cost from the start (Dijkstra's distance) and h(v) is a **heuristic**: an estimate of the cost still to go from v to the goal. Cells that look closer to the goal are explored first.

The estimate must never **overestimate** the true remaining cost; such a heuristic is called **admissible**, and with it A\* still finds a shortest path. For grids:

- moving in 4 directions with cost 1: the **Manhattan distance** |r − gr| + |c − gc|;
- moving in 8 directions with costs 1 and √2: the **octile distance** max(dr, dc) + (√2 − 1) × min(dr, dc).

Two extremes show the trade-off. With h = 0, A\* is exactly Dijkstra's algorithm. With a perfect h, it walks straight down the shortest path and explores nothing else. A good heuristic sits in between: correct, and far less work. (If h is also **consistent**, h(u) ≤ cost(u, v) + h(v) for every move, which Manhattan distance is, each cell is settled once, just as in Dijkstra's algorithm.)

Both find the same length; A\* expands a fraction of the cells. Notice the **tie-break**: on an open grid, every cell inside the rectangle between start and goal has the same f, so the second item in each heap entry decides. Pushing `-(d + 1)` prefers the **largest** g among equal f, the cell furthest along, and A\* heads straight for the goal; preferring the smallest g would make it expand the whole rectangle.

## From both ends

On an unweighted graph, a BFS that reaches distance d explores roughly b^d vertices if each has about b neighbours. Searching from **both** ends at once and stopping when the two searches meet needs only about 2 × b^(d/2), which for large d is dramatically less. **Bidirectional BFS** expands one level at a time from whichever side has the smaller frontier, and stops as soon as a newly reached vertex has been seen by the other side. (It needs to be able to step backwards from the goal, which is easy for undirected graphs and puzzles with reversible moves.)

## Cells plus state

Some grid problems can't be solved by searching cells alone, because what you can do next depends on more than where you are:

- **Keys and doors**: a door `A` opens only if you've picked up key `a`. The state is `(cell, keys held)`. With up to a dozen keys, store the keys as a **bitmask**, an integer whose bit i is 1 if key i is held: `keys | (1 << i)` adds key i, `keys & (1 << i)` tests it.
- **Direction**: if turning costs time (a robot, a car), the state is `(cell, facing)`, and moving straight on is cheap while turning costs more.
- **Limited resources**: walls you may break through (at most k), fuel, a time of day: add the remaining amount to the state.

The state space is larger (cells × key sets, cells × 4 directions), but each state is still visited once, so BFS or Dijkstra's algorithm remains efficient. Week 50's lesson again: choose vertices so that the next move depends only on the vertex.

The direct way is barred by door A; the search goes down and along to fetch key a (6 steps), comes back to the start (6 more), then through the door to the exit (4): 16 steps.

## Which search?

| the grid | use |
|---|---|
| every step costs the same | BFS |
| a few costs of 0 and 1 | 0-1 BFS (Week 50 Day 2) |
| varied non-negative costs | Dijkstra's algorithm |
| varied costs, one known goal, a good estimate of distance | A\* |
| unweighted, huge, reversible moves | bidirectional BFS |
| moves depend on keys, direction, resources | any of the above on (cell, state) |
| nearest of many sources | multi-source BFS or Dijkstra |

## Practice

## Quiz

## Daily challenge

## Before you go

Draw a small maze on squared paper with a key and a locked door, and solve it by hand thinking in states: "where am I, and which keys do I hold?" Tomorrow is the Forge: a delivery planner that brings together this week's weighted graph algorithms.